In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [4]:
##loading the trained model
model = load_model('churn_model.h5')

##loading the scaler
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
##loading the encoder
with open('onehot_encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

In [5]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [6]:
## one hot encoding the categorical features
geography_encoded = encoder.transform([[input_data['Geography']]]).toarray()
geography_df = pd.DataFrame(geography_encoded, columns=encoder.get_feature_names_out(['Geography']))
geography_df

g:\annproject\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [7]:
input_data_df = pd.DataFrame([input_data])

In [8]:
## encoding the gender feature
input_data_df['Gender'] = label_encoder.transform(input_data_df['Gender'])

In [9]:
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [10]:
##concatenating the one hot encoded geography features with the input data and dropping the original Geography column
input_data_df = pd.concat([input_data_df, geography_df], axis=1)
input_data_df = input_data_df.drop('Geography', axis=1)

In [11]:
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [12]:
##scaling the input data
scaled_input_data = scaler.transform(input_data_df)
scaled_input_data

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [13]:
##predicting the churn probability
churn_probability = model.predict(scaled_input_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step


In [14]:
churn_probability

array([[0.03413426]], dtype=float32)

In [15]:
print(f"Churn Probability: {churn_probability[0][0]:.4f}")

Churn Probability: 0.0341


In [16]:
if churn_probability[0][0] > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is unlikely to churn.")

The customer is unlikely to churn.
